# Transformers

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2

In [2]:
from unsloth import FastModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
from unsloth import FastModel
import torch

model_id = "mayurmadnani/gemma-3-270m-microfables"

model, tokenizer = FastModel.from_pretrained(
    model_name = model_id,
    max_seq_length = 2048,
    load_in_4bit = False,
    load_in_8bit = False,
    dtype = None, # None for auto detection
)

# Enable native 2x faster inference
FastModel.for_inference(model)

print(f"Model and tokenizer loaded successfully from {model_id}")

==((====))==  Unsloth 2026.2.1: Fast Gemma3 patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

Model and tokenizer loaded successfully from mayurmadnani/gemma-3-270m-microfables


In [4]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma3",
)

In [5]:
from transformers import TextStreamer
import torch

model = model.to(torch.float32)

messages = [
    {"role": "user", "content": "Write a short story about a brave knight."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

text_streamer = TextStreamer(tokenizer)

_ = model.generate(
    input_ids = inputs,
    streamer = text_streamer,
    max_new_tokens = 1024,
    use_cache = True
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<bos><start_of_turn>user
Write a short story about a brave knight.<end_of_turn>
<start_of_turn>model
At court, a knight stood bold and strong. His sword flashed in the light, and he rode bravely through battle. Enemies fell, and the knight’s courage saved many lives. He may be young, but his heart was brave. The people cheered for him.<end_of_turn>


# Ollama

In [6]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (464 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently i

In [7]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [8]:
import subprocess
import time

# Start ollama serve in the background
ollama_process = subprocess.Popen(['ollama', 'serve'])

# Give the processes a moment to start
time.sleep(5)

In [9]:
!ollama pull mayurmadnani/gemma-3-270m-microfables:latest

In [10]:
!ollama ls

NAME                                            ID              SIZE      MODIFIED               
mayurmadnani/gemma-3-270m-microfables:latest    8ee55d1341b3    291 MB    Less than a second ago    


In [11]:
!ollama run mayurmadnani/gemma-3-270m-microfables:latest "Write a short story about a brave knight."

A knight faced down a dragon at the castle gates. The dragon roared and lunged, but the knight held his ground. He rode faster and faster, and the dragon fell back down. The knight crossed the sea, proud and strong.



In [12]:
!curl http://localhost:11434/api/generate -d '{ \
  "model": "mayurmadnani/gemma-3-270m-microfables:latest", \
  "prompt": "Write a short story about a brave knight.", \
  "stream": false \
}'

{"model":"mayurmadnani/gemma-3-270m-microfables:latest","created_at":"2026-02-17T09:33:51.448833765Z","response":"King Alaric faced down a dragon with shining armor. The dragon roared and tried to crush the knight. Alaric raised his sword, and the knight charged forward. They fought hard, but the dragon held firm. With a final roar, the knight reached the middle of the beast. The dragon fell, and the king crowned him king again.","done":true,"done_reason":"stop","context":[105,2364,107,6974,496,2822,3925,1003,496,36711,52482,236761,106,107,105,4368,107,107,29776,1429,80775,17175,1679,496,25800,607,36489,38119,236761,669,25800,149193,532,6956,531,47585,506,52482,236761,1429,80775,8675,914,26114,236764,532,506,52482,11055,4448,236761,2195,25876,2651,236764,840,506,25800,4247,6218,236761,3227,496,1626,96887,236764,506,52482,8452,506,6029,529,506,42239,236761,669,25800,11561,236764,532,506,9615,75216,1515,9615,1570,236761],"total_duration":1189716059,"load_duration":471909888,"prompt_eval_